In [2]:
import pandas as pd
import altair as alt
import numpy as np

def analyze_interactive_patterns(file_path):
    # ---------------------------------------------------------
    # 1. DATA PREP (Standard)
    # ---------------------------------------------------------
    df = pd.read_csv(file_path)
    
    date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        
    df = df.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date', 'price', 'freight_value'])
    
    # Feature Engineering
    df['actual_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.total_seconds() / (24*3600)
    df['buffer_days'] = (df['order_estimated_delivery_date'] - df['order_delivered_customer_date']).dt.total_seconds() / (24*3600)
    df['freight_ratio'] = df['freight_value'] / df['price'].replace(0, np.nan)
    df['month_year'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

    # ---------------------------------------------------------
    # 2. AGGREGATIONS (With Dummy Groups for Line Fix)
    # ---------------------------------------------------------

    # A. Logistics Matrix
    state_stats = df.groupby('customer_state').agg({
        'actual_days': 'mean', 'buffer_days': 'mean', 'order_id': 'count'
    }).rename(columns={'order_id': 'order_count'}).reset_index()
    state_stats = state_stats[state_stats['order_count'] > 50]
    state_stats['vol_rank'] = state_stats['order_count'].rank(pct=True)

    # B. Poor Tax & Premium Illusion
    df['price_decile'] = pd.qcut(df['price'], q=10, labels=False, duplicates='drop')
    decile_stats = df.groupby('price_decile').agg({
        'freight_ratio': 'mean', 'actual_days': 'mean', 'price': 'mean'
    }).reset_index()
    decile_stats['group_id'] = 1  # <--- FIX: Dummy group for line connection

    # C. Trust Curve
    df_sorted = df.sort_values(['customer_unique_id', 'order_purchase_timestamp'])
    customer_history = df_sorted.groupby('customer_unique_id').agg({
        'actual_days': 'first', 'order_id': 'nunique'
    }).reset_index()
    customer_history['is_retained'] = customer_history['order_id'] > 1
    
    customer_history = customer_history[customer_history['actual_days'].between(0, 45)]
    customer_history['delivery_bin'] = pd.cut(customer_history['actual_days'], bins=range(0, 46, 3))
    customer_history['bin_label'] = customer_history['delivery_bin'].apply(lambda x: f"{int(x.left)}-{int(x.right)}")
    customer_history['bin_start'] = customer_history['delivery_bin'].apply(lambda x: x.left)
    
    retention_stats = customer_history.groupby(['bin_label', 'bin_start']).agg({
        'is_retained': 'mean'
    }).reset_index().sort_values('bin_start')
    retention_stats['group_id'] = 1 # <--- FIX: Dummy group for line connection

    # D. Seasonal Collapse
    seasonal_stats = df.groupby('month_year').agg({
        'actual_days': 'mean', 'order_id': 'count'
    }).rename(columns={'order_id': 'order_count'}).reset_index()
    seasonal_stats['group_id'] = 1 # <--- FIX: Dummy group for line connection

    # ---------------------------------------------------------
    # 3. INTERACTIVE VISUALIZATIONS (Altair)
    # ---------------------------------------------------------
    
    # Chart 1: Logistics Matrix
    chart1 = alt.Chart(state_stats).mark_circle(size=120).encode(
        x=alt.X('actual_days:Q', title='Actual Days'),
        y=alt.Y('buffer_days:Q', title='Buffer Days'),
        color=alt.Color('vol_rank:Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title='Vol Rank')),
        tooltip=[
            alt.Tooltip('customer_state', title='State'),
            alt.Tooltip('actual_days', format='.1f', title='Actual (Days)'),
            alt.Tooltip('buffer_days', format='.1f', title='Buffer (Days)')
        ]
    ).properties(title='A. Logistics Matrix (Hover for State)').interactive()

    # Chart 2: Poor Tax
    chart2 = alt.Chart(decile_stats).mark_bar().encode(
        x=alt.X('price_decile:O', title='Price Decile'),
        y=alt.Y('freight_ratio:Q', title='Freight Ratio'),
        color=alt.Color('freight_ratio:Q', scale=alt.Scale(scheme='magma')),
        tooltip=[
            alt.Tooltip('price_decile', title='Decile'),
            alt.Tooltip('freight_ratio', format='.2f', title='Freight Ratio')
        ]
    ).properties(title='B. Poor Tax').interactive()

    # Chart 3: Trust Curve (Line) - FIX IMPLEMENTED HERE
    line3 = alt.Chart(retention_stats).mark_line(point=True, color='#c0392b').encode(
        x=alt.X('bin_label:N', sort=alt.SortField('bin_start'), title='Delivery Days (Bin)'),
        y=alt.Y('is_retained:Q', title='Retention Prob'),
        detail='group_id', # <--- Forces connection across Nominal axis
        tooltip=[
            alt.Tooltip('bin_label', title='Days'),
            alt.Tooltip('is_retained', format='.1%', title='Retention')
        ]
    ).properties(title='C. Trust Curve').interactive()

    # Chart 4: Premium Illusion (Line) - FIX IMPLEMENTED HERE
    line4 = alt.Chart(decile_stats).mark_line(point=True, color='#2980b9').encode(
        x=alt.X('price_decile:O', title='Price Decile'),
        y=alt.Y('actual_days:Q', title='Actual Days'),
        detail='group_id', # <--- Forces connection across Ordinal axis
        tooltip=[
            alt.Tooltip('price_decile', title='Decile'),
            alt.Tooltip('actual_days', format='.1f', title='Avg Days')
        ]
    ).properties(title='D. Premium Illusion').interactive()

    # Chart 5: Seasonal Collapse (Line) - FIX IMPLEMENTED HERE
    line5 = alt.Chart(seasonal_stats).mark_line(point=True, color='#8e44ad').encode(
        x=alt.X('month_year:N', title='Month'),
        y=alt.Y('actual_days:Q', title='Avg Days'),
        detail='group_id', # <--- Forces connection across Nominal string dates
        tooltip=[
            alt.Tooltip('month_year', title='Date'),
            alt.Tooltip('actual_days', format='.1f', title='Avg Days')
        ]
    ).properties(title='E. Seasonal Collapse', width=800).interactive()

    # Assemble Dashboard
    row1 = alt.hconcat(chart1, chart2)
    row2 = alt.hconcat(line3, line4)
    dashboard = alt.vconcat(row1, row2, line5).resolve_scale(color='independent')
    
    return dashboard

dashboard = analyze_interactive_patterns('Brazilian E-Commerce Public Dataset by Olist.csv')
# dashboard.save('interactive_analysis.json')
dashboard

C:\Users\i232623\AppData\Local\Temp\ipykernel_15568\269366626.py:53: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  retention_stats = customer_history.groupby(['bin_label', 'bin_start']).agg({
C:\Users\i232623\AppData\Local\Temp\ipykernel_15568\269366626.py:131: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dashboard = analyze_interactive_patterns('Brazilian E-Commerce Public Dataset by Olist.csv')
C:\Users\i232623\AppData\Local\Temp\ipykernel_15568\269366626.py:131: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly 

alt.VConcatChart(...)